In [1]:
import pandas as pd
from argopy import DataFetcher
import xarray as xr

argo_loader = DataFetcher()
df = argo_loader.region([32, 80, 8, 26, 0, 2000, '2023-03-01', '2023-03-31']).to_xarray()
df = df.to_dataframe().reset_index()
columns_to_keep = [
    'PLATFORM_NUMBER', 
    'CYCLE_NUMBER', 
    'LATITUDE', 
    'LONGITUDE', 
    'TIME',
    'PRES', 
    'TEMP', 
    'PSAL'
]
# df.dropna(inplace=True)
# df = df[columns_to_keep]
df.columns
# df.head()

/home/rohith/Projects/Poseidon-RAG/Backend/.venv/bin/python: No module named pip


Index(['N_POINTS', 'CYCLE_NUMBER', 'DATA_MODE', 'DIRECTION', 'PLATFORM_NUMBER',
       'POSITION_QC', 'PRES', 'PRES_ERROR', 'PRES_QC', 'PSAL', 'PSAL_ERROR',
       'PSAL_QC', 'TEMP', 'TEMP_ERROR', 'TEMP_QC', 'TIME_QC', 'LATITUDE',
       'LONGITUDE', 'TIME'],
      dtype='str')

In [2]:
df.info()

<class 'pandas.DataFrame'>
Index: 89679 entries, 0 to 94699
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   PLATFORM_NUMBER  89679 non-null  int64         
 1   CYCLE_NUMBER     89679 non-null  int64         
 2   LATITUDE         89679 non-null  float64       
 3   LONGITUDE        89679 non-null  float64       
 4   TIME             89679 non-null  datetime64[ns]
 5   PRES             89679 non-null  float32       
 6   TEMP             89679 non-null  float32       
 7   PSAL             89679 non-null  float32       
dtypes: datetime64[ns](1), float32(3), float64(2), int64(2)
memory usage: 5.1 MB


In [3]:
df['TIME'].dt.month.nunique()

1

In [4]:
import os
import sqlite3

conn = sqlite3.connect(os.path.join(os.getcwd(),'data','argo_data.db'))
df.to_sql('measurements',con=conn,index=False,if_exists='replace')

89679

In [11]:
query = '''
    SELECT 
        Platform_number,
        COUNT(*) as total_readings,
        ROUND(MIN(temp), 2) as min_temp,
        ROUND(MAX(temp), 2) as max_temp,
        ROUND(AVG(latitude), 4) as avg_lat,
        ROUND(AVG(longitude), 4) as avg_lon
    FROM measurements
    GROUP BY Platform_number
'''

query ="""
        SELECT DISTINCT(Date(time)) FROM measurements 
        WHERE platform_number = '2902270'
        """

cursor = conn.cursor()
cursor.execute(query)

rows = cursor.fetchall()

print(rows)

[('2023-03-01',), ('2023-03-12',), ('2023-03-22',)]
